In [1]:
from pathlib import Path
import os, json, hashlib
import chromadb
from openai import OpenAI

KB_SOURCE_JSONL = Path(r"D:\shixi_agent\黄剑企业微信导出\out4\kb_clean.jsonl")
KB_CHROMA_DIR = r"D:\shixi_agent\黄剑企业微信导出\out4_chroma"  # 你也可以换成别的目录
COLLECTION_NAME = "hj_wecom_kb"

EMBED_API_BASE = "https://api.deepseek.com/v1"
EMBED_API_KEY = os.environ.get("EMBED_API_KEY", "").strip()  # 建议在环境变量或 .env 里
EMBED_MODEL = os.environ.get("EMBED_MODEL", "").strip()      # 例如 deepseek-embedding（以你实际为准）

assert KB_SOURCE_JSONL.is_file(), f"找不到: {KB_SOURCE_JSONL}"
assert EMBED_API_KEY, "请先设置环境变量 EMBED_API_KEY"
assert EMBED_MODEL, "请先设置环境变量 EMBED_MODEL（embedding模型名）"

client = OpenAI(api_key=EMBED_API_KEY, base_url=EMBED_API_BASE)

chroma = chromadb.PersistentClient(path=KB_CHROMA_DIR)
col = chroma.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

def make_id(rec):
    # 尽量稳定：优先用已有 hash；否则用 question+answer hash
    h = (rec.get("hash") or "").strip()
    if h:
        return h
    s = (rec.get("question","") + "\n" + rec.get("answer","")).encode("utf-8", errors="ignore")
    return hashlib.md5(s).hexdigest()

def to_doc(rec):
    # 把 Q/A 拼成一个可检索文本（后续检索用 query 查这个）
    q = (rec.get("question") or "").strip()
    a = (rec.get("answer") or "").strip()
    return f"Q: {q}\nA: {a}".strip()

rows = []
with KB_SOURCE_JSONL.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

len(rows), rows[0].keys()

AssertionError: 请先设置环境变量 EMBED_API_KEY

In [ ]:
BATCH = 128

def embed_texts(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]

added = 0

for i in range(0, len(rows), BATCH):
    batch = rows[i:i+BATCH]
    ids = [make_id(r) for r in batch]
    docs = [to_doc(r) for r in batch]
    metas = [{
        "source_file": r.get("source_file",""),
        "q_speaker": r.get("q_speaker",""),
        "q_ts": r.get("q_ts",""),
    } for r in batch]

    embs = embed_texts(docs)
    col.upsert(ids=ids, documents=docs, metadatas=metas, embeddings=embs)
    added += len(batch)

print("Done. upserted =", added)
print("Chroma dir =", KB_CHROMA_DIR)
print("collection count =", col.count())

In [ ]:
query = "企业微信聊天记录怎么导出"
q_emb = embed_texts([query])[0]
res = col.query(query_embeddings=[q_emb], n_results=5)

for d, m in zip(res["documents"][0], res["metadatas"][0]):
    print("----", m.get("source_file",""))
    print(d[:300])

In [2]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ["EMBED_API_KEY"],
    base_url=os.environ.get("EMBED_API_BASE", "https://api.deepseek.com/v1"),
)

EMBED_MODEL = os.environ["EMBED_MODEL"]

def embed_texts(texts):
    try:
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
        return [d.embedding for d in resp.data]
    except Exception as e:
        print("EXC_TYPE:", type(e))
        # openai-python 的异常通常带 response
        resp = getattr(e, "response", None)
        if resp is not None:
            print("STATUS:", resp.status_code)
            try:
                print("BODY:", resp.text[:2000])
            except Exception:
                pass
            try:
                print("HEADERS:", dict(resp.headers))
            except Exception:
                pass
        raise

# 最小调用
embed_texts(["hello"])

KeyError: 'EMBED_API_KEY'

In [3]:
from dotenv import load_dotenv
load_dotenv()  # 默认读取当前工作目录下的 .env

True

In [4]:
import os
print("EMBED_API_BASE =", os.environ.get("EMBED_API_BASE"))
print("EMBED_MODEL =", os.environ.get("EMBED_MODEL"))
print("EMBED_API_KEY exists =", bool(os.environ.get("EMBED_API_KEY")))

EMBED_API_BASE = https://dashscope.aliyuncs.com/compatible-mode/v1
EMBED_MODEL = text-embedding-v4
EMBED_API_KEY exists = True


In [5]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ["EMBED_API_KEY"],
    base_url=os.environ.get("EMBED_API_BASE", "https://api.deepseek.com/v1"),
)

resp = client.embeddings.create(
    model=os.environ["EMBED_MODEL"],
    input=["hello"],
)
print(len(resp.data[0].embedding), resp.data[0].embedding[:5])

1024 [0.033521413803100586, 0.0039935060776770115, 0.00853331945836544, 0.0350729264318943, 0.05371293053030968]


In [18]:
import os
print("EMBED_API_BASE =", os.environ.get("EMBED_API_BASE"))
print("EMBED_MODEL    =", os.environ.get("EMBED_MODEL"))

EMBED_API_BASE = https://api.deepseek.com/v1
EMBED_MODEL    = deepseek-embedding


In [13]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ["EMBED_API_KEY"],
    base_url=os.environ.get("EMBED_API_BASE", "https://api.deepseek.com/v1"),
)

try:
    client.embeddings.create(model=os.environ["EMBED_MODEL"], input=["hello"])
except Exception as e:
    print("EXC_TYPE:", type(e))
    resp = getattr(e, "response", None)
    if resp is not None:
        print("STATUS:", resp.status_code)
        try:
            print("BODY:", resp.text[:2000])
        except Exception:
            pass
        try:
            print("HEADERS:", dict(resp.headers))
        except Exception:
            pass
    raise

EXC_TYPE: <class 'openai.NotFoundError'>
STATUS: 404
BODY: 
HEADERS: {'content-length': '0', 'connection': 'keep-alive', 'date': 'Wed, 08 Apr 2026 10:12:48 GMT', 'server': 'elb', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-credentials': 'true', 'x-ds-trace-id': '6695687f838efc11bcef689c9cd70376', 'strict-transport-security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-cache': 'Error from cloudfront', 'via': '1.1 e661d3bc2cbf326fe5efbcf97cecea8c.cloudfront.net (CloudFront)', 'x-amz-cf-pop': 'SEA73-P3', 'x-amz-cf-id': '1kDwB0Xpbi5AA4Wp1dyDm8YGjoSlBwO-0-K-K3NwbmYiBRetttwCcA=='}


NotFoundError: Error code: 404

In [14]:
from openai import OpenAI
import os

c = OpenAI(api_key=os.environ["EMBED_API_KEY"], base_url=os.environ.get("EMBED_API_BASE","https://api.deepseek.com/v1"))

try:
    ms = c.models.list()
    print("models_count =", len(ms.data))
    print([m.id for m in ms.data][:50])
except Exception as e:
    resp = getattr(e, "response", None)
    print("models.list failed:", type(e))
    if resp is not None:
        print("STATUS:", resp.status_code)
        try:
            print("BODY:", resp.text[:2000])
        except Exception:
            pass
    raise

models_count = 2
['deepseek-chat', 'deepseek-reasoner']


In [17]:
import httpx, os

base = "https://api.deepseek.com/v1"
h = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}

for path in ["/models", "/chat/completions", "/embeddings"]:
    r = httpx.get(base + path, headers=h, timeout=20.0) if path=="/models" else httpx.post(base+path, headers={**h,"Content-Type":"application/json"}, json={"model":"deepseek-chat","messages":[{"role":"user","content":"hi"}]} if path=="/chat/completions" else {"model":"deepseek-embedding","input":["hello"]}, timeout=20.0)
    print(path, r.status_code, (r.text or "")[:200])

/models 200 {"object":"list","data":[{"id":"deepseek-chat","object":"model","owned_by":"deepseek"},{"id":"deepseek-reasoner","object":"model","owned_by":"deepseek"}]}
/chat/completions 200 {"id":"83771ffb-185a-4b60-a95d-fad086458135","object":"chat.completion","created":1775644444,"model":"deepseek-chat","choices":[{"index":0,"message":{"role":"assistant","content":"你好！👋 很高兴见到你！\n\n有什么我
/embeddings 404 


In [6]:
from dotenv import load_dotenv
load_dotenv(r"C:\Users\10409\Documents\work\shixi_agent\medical_agent_demo\.env")
import os
from openai import OpenAI
client = OpenAI(
    api_key=os.environ["EMBED_API_KEY"],
    base_url=os.environ["EMBED_API_BASE"],  # dashscope compatible-mode/v1
)
resp = client.embeddings.create(
    model=os.environ["EMBED_MODEL"],  # text-embedding-v4
    input=["hello"]
)
print(len(resp.data[0].embedding), resp.data[0].embedding[:5])

1024 [0.033521413803100586, 0.0039935060776770115, 0.00853331945836544, 0.0350729264318943, 0.05371293053030968]


In [7]:
import json
from pathlib import Path

p = Path(r"D:\shixi_agent\黄剑企业微信导出\out4\kb_clean.jsonl")
rows = []
with p.open("r", encoding="utf-8") as f:
    for _ in range(3):
        rows.append(json.loads(next(f)))

texts = [f"Q: {r['question']}\nA: {r['answer']}" for r in rows]
resp = client.embeddings.create(model=os.environ["EMBED_MODEL"], input=texts)
print("batch ok, n=", len(resp.data))

batch ok, n= 3
